# Stage 3: Singlish Normalisation

### Project Structure

```bash
CS5246Project/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                               
    ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb          
    ├─ Stage_2_POS_and_NER_Tagging.ipynb                        
-----------------------------------------------------------------            
│   ├─ Stage_3_Singlish_Normalisation.ipynb                     │
-----------------------------------------------------------------
    ├─ Stage_4_Singlish_to_English_Conversion.ipynb     
    ├─ Stage_5_Common_Normalisation.ipynb
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

## To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect import-ipynb tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.1 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=cdbd0bd04afef71d1d6bb376cb07d227f755d6079bd25f760cbb50e87e0e7e84
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


### Change Directory

In [ ]:
# Path
# ----
FOLDER_PATH: str = "/content/drive/MyDrive/CS5246Project/"

import os
os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import gc
import regex
import spacy
import import_ipynb
import pandas as pd
from tqdm import tqdm
from Stage_2_POS_and_NER_Tagging import pos_tagger
from Stage_0_Introduction import display_first_text_column_head, save_dfs_to_csv, save_single_df_to_csv, output_base_dir

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.1 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Singlish Normalisation

### POS Tagging Test

In [ ]:
sentences = [
    "Christian Ho sets sights on F3 after winning race",
    "trash to treasure what would you do with an essence of chicken box make chairs lor",
    "Sian la work tomorrow...",
    "k pop group le sserafim learn kopi ordering vocabulary for s pore concert",
    "how can lei",
    "ba in product design or ba in industrial design",
    "m sian man brings s 300 00 into s pore to gamble doesn t declare it because it s troublesome fined s 10 00"
]

for i, sent in enumerate(sentences):
    tags = pos_tagger.tag_text(sent)
    print(f"\nSentence {i+1}: {sent}")
    print("Tokens and tags:")
    for token_info in tags:
        print(f"  {token_info['token']:15} POS: {token_info['pos']:6} Tag: {token_info['tag']}")



Sentence 1: Christian Ho sets sights on F3 after winning race
Tokens and tags:
  Christian       POS: PROPN  Tag: NNP
  Ho              POS: PROPN  Tag: NNP
  sets            POS: VERB   Tag: VBZ
  sights          POS: NOUN   Tag: NNS
  on              POS: ADP    Tag: IN
  F3              POS: PROPN  Tag: NNP
  after           POS: ADP    Tag: IN
  winning         POS: VERB   Tag: VBG
  race            POS: NOUN   Tag: NN

Sentence 2: trash to treasure what would you do with an essence of chicken box make chairs lor
Tokens and tags:
  trash           POS: VERB   Tag: VB
  to              POS: PART   Tag: TO
  treasure        POS: VERB   Tag: VB
  what            POS: PRON   Tag: WP
  would           POS: AUX    Tag: MD
  you             POS: PRON   Tag: PRP
  do              POS: VERB   Tag: VB
  with            POS: ADP    Tag: IN
  an              POS: DET    Tag: DT
  essence         POS: NOUN   Tag: NN
  of              POS: ADP    Tag: IN
  chicken         POS: NOUN   Tag: NN
  

In [ ]:
%cd /content/drive/MyDrive/CS5246Project/data/

/content/drive/.shortcut-targets-by-id/1w4egvqzmOPu5lqEo9ID8hi6Na3YQatxk/CS5246Project/data


### Load Data

In [ ]:
# ---------------
# Load Clean Data
# ---------------
df_posts = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/PostVault.csv', keep_default_na=False)
df_comments = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/CommentVault.csv', keep_default_na=False)

/tmp/ipykernel_2874/4176693567.py:4: DtypeWarning: Columns (20,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df_posts = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/PostVault.csv', keep_default_na=False)


### Singlish Normalization with POS

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
# Enable the tagger pipeline component, which assigns part-of-speech tags to tokens.
nlp.enable_pipe("tagger")

regex_map = []

mapping_file = '/content/drive/MyDrive/CS5246Project/utilities/singlish_regex_to_text.txt'
with open(mapping_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip() 
        if not line or line.startswith('#'):
            continue
        parts = line.rsplit(',', 1)
        if len(parts) != 2:
            print("Skipping invalid line:", line)
            continue
        pattern, replacement = parts
        regex_map.append((pattern.strip(), replacement.strip()))

regex_map_compiled = [(regex.compile(pat, flags=regex.IGNORECASE), repl) for pat, repl in regex_map]

def normalize_token(word, pos=None):
    if pos == "PROPN":
        return word
    new_word = word
    for pattern, replacement in regex_map_compiled:
        new_word = pattern.sub(replacement, new_word)
    return new_word

def normalize_doc(doc):
    text = doc.text

    for tok in reversed(list(doc)):
        if tok.is_space or tok.pos_ == "PROPN":
            continue
        old = tok.text 
        new = normalize_token(old, tok.pos_) 
        
        if old != new:
            start, end = tok.idx, tok.idx + len(tok.text)
            text = text[:start] + new + text[end:]
    return text

def normalize_columns(df, text_columns, nlp, normalize_doc, batch_size=32, verbose=True):
    for col in text_columns:
        if col not in df.columns:
            continue

        normalized_col = f"singlish_normalized_{col.replace('cleaned_', '')}"

        texts = df[col].astype(str).tolist()
        normalized_texts = [] 

        for j in range(0, len(texts), batch_size):
            batch_texts = texts[j:j + batch_size]

            docs = list(nlp.pipe(batch_texts, batch_size=batch_size))
            batch_normalized = [normalize_doc(doc) for doc in docs]
            normalized_texts.extend(batch_normalized) 

            del docs
            del batch_normalized
            gc.collect()

        df[normalized_col] = normalized_texts

        if verbose:
            original_col_for_compare = df[col].astype(str).fillna('')
            normalized_col_for_compare = df[normalized_col].astype(str).fillna('')

            changed_rows = df[
                original_col_for_compare.str.replace(" ", "", regex=False) !=
                normalized_col_for_compare.str.replace(" ", "", regex=False)
            ]

            for idx, before, after in zip(
                changed_rows.index,
                changed_rows[col],
                changed_rows[normalized_col]
            ):
                print(f"ROW    : {idx}")
                print("BEFORE :", before)
                print("AFTER  :", after)
                print("-" * 50)

        del texts
        del normalized_texts
        gc.collect()

    return df

#### PostVault

In [ ]:
text_columns = ['cleaned_title', 'cleaned_content']

df_posts = normalize_columns(df_posts, text_columns, nlp, normalize_doc, batch_size=32, verbose=True)


ROW    : 336
BEFORE : m sian man 38 pretends to be us sugar daddy forces 3 s pore based women to have sex with driver who was also him
AFTER  : m sienz man 38 pretends to be us sugar daddy forces 3 s pore based women to have sex with driver who was also him
--------------------------------------------------
ROW    : 604
BEFORE : trash wastebasket to treasure trophy what would you do with an essence of chicken box make chairs lor chair face with tears of joy red heart
AFTER  : trash wastebasket to treasure trophy what would you do with an essence of chicken box make chairs loh chair face with tears of joy red heart
--------------------------------------------------
ROW    : 708
BEFORE : n m ch n s you l m t trong nh ng lo i n m c ng you v kh i you tr nh t v i c c m n m n m s you d i da vi c lo i b ho n to n ch ng i h i ph ng ph p i you tr chuy n s you v ki n tr b i vi t n y s chia s c c ph ng ph p i you tr n m ch n s you hi you qu t c ch i you tr t i nh n c c li you
AFTER  : n m ch n s 

#### CommentVault

In [ ]:
text_columns = ['cleaned_text']

df_comments = normalize_columns(df_comments, text_columns, nlp, normalize_doc, batch_size=32, verbose=True)

### Save Files

#### PostVault

In [ ]:
df_posts.columns

Index(['id', 'title', 'content', 'cleaned_title', 'cleaned_content', 'score',
       'upvote_ratio', 'num_comments', 'created_utc', 'subreddit_id',
       'has_emoji', 'year', 'month', 'day_of_week', 'hour', 'score_bucket',
       'text_length', 'word_count', 'singlish_normalized_title',
       'singlish_normalized_content', 'singlish_count',
       'english_converted_title', 'english_converted_content',
       'expanded_title', 'expanded_content', 'demojized_title',
       'demojized_content', 'spellchecked_title', 'spellchecked_content',
       'lemmatized_title', 'lemmatized_content', 'tfidf_cluster'],
      dtype='object')

In [ ]:
# ---------
# PostVault
# ---------

display(df_posts.head())
print("-----------")
print("Posts Shape")
print("-----------")
print(df_posts.shape)
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,spellchecked_title,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,fantasy meets whimsy at medieval themed renais...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,pair of adjoining ground floor hub shops in pa...,pair adjoining ground floor hdb shop pasir ri ...,,,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,property remaining tenure 63 year available sa...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,video call scam targeting elderly,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,90 year old mum got video call someone full po...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,woodlands checkpoint officer dragged by motori...,woodland checkpoint officer dragged motorist f...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,,,,,,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,bangkok post singapore a transit point for up ...,bangkok post singapore transit point uk bound ...,,,,,,,,1.0


-----------
Posts Shape
-----------
(31957, 32)
Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


#### CommentVault

In [ ]:
# ------------
# CommentVault
# ------------

display(df_comments.head())
print("--------------")
print("Comments Shape")
print("--------------")
print(df_comments.shape)
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")